# Bygg komplett FPL-datasett for alle sesonger

Notebooken leser hver sesongs `gws/merged_gw.csv`, slår sesongene sammen, legger til motstandernavn og beregner lagstatistikk. Originalfilene endres ikke.

Resultatet lagres som `data-source/data/cleaned_merged_seasons_team_aggregated_expanded.csv`.

In [ ]:
%pip install -q pandas

## 1. Finn data og sesonger

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)

candidates = [
    Path.cwd() / "data-source" / "data",
    Path.cwd().parent / "data-source" / "data",
]
DATA_DIR = next((path for path in candidates if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Fant ikke data-source/data")

seasons = sorted(
    path.name
    for path in DATA_DIR.iterdir()
    if path.is_dir()
    and path.name[:4].isdigit()
    and (path / "gws" / "merged_gw.csv").exists()
)

print("Datamappe:", DATA_DIR)
print("Sesonger som skal inkluderes:", seasons)

## 2. Last inn og slå sammen gameweek-data

Kolonnene har endret seg mellom sesongene. `concat(..., sort=False)` beholder unionen av alle kolonner og fyller manglende historiske felt med `NaN`.

In [ ]:
frames = []
load_report = []

for season in seasons:
    path = DATA_DIR / season / "gws" / "merged_gw.csv"
    try:
        season_data = pd.read_csv(path)
    except UnicodeDecodeError:
        season_data = pd.read_csv(path, encoding="latin-1")

    season_data["season_x"] = season
    if "GW" not in season_data.columns and "round" in season_data.columns:
        season_data["GW"] = season_data["round"]

    frames.append(season_data)
    load_report.append({
        "season": season,
        "rows": len(season_data),
        "columns": len(season_data.columns),
        "gameweeks": season_data["GW"].nunique(),
    })

all_seasons = pd.concat(frames, ignore_index=True, sort=False)
all_seasons["kickoff_time"] = pd.to_datetime(
    all_seasons["kickoff_time"], errors="coerce", utc=True
)

# Kildedataene har enkelte identiske rader og noen utsatte kamper registrert
# både på opprinnelig og ny dato. Behold siste rad per spiller og fixture.
player_fixture_key = ["season_x", "element", "fixture"]
rows_before_deduplication = len(all_seasons)
all_seasons = (
    all_seasons.sort_values(player_fixture_key + ["kickoff_time"], na_position="first")
    .drop_duplicates(player_fixture_key, keep="last")
    .reset_index(drop=True)
)
print(f"Fjernet {rows_before_deduplication - len(all_seasons)} duplikatrader")

# De eldste sesongene mangler spillerens lag. I hver kamp er spillerens
# lag den andre unike opponent_team-ID-en som finnes i samme fixture.
participants = all_seasons[["season_x", "fixture", "opponent_team"]].drop_duplicates()
own_team_lookup = participants.merge(
    participants, on=["season_x", "fixture"], suffixes=("", "_other")
)
own_team_lookup = own_team_lookup.loc[
    own_team_lookup["opponent_team"].ne(own_team_lookup["opponent_team_other"]),
    ["season_x", "fixture", "opponent_team", "opponent_team_other"],
].rename(columns={"opponent_team_other": "derived_team_id"})
own_team_lookup = own_team_lookup.drop_duplicates(
    ["season_x", "fixture", "opponent_team"]
)
all_seasons = all_seasons.merge(
    own_team_lookup, on=["season_x", "fixture", "opponent_team"], how="left",
    validate="many_to_one",
)

# Lag en sesongspesifikk ID->navn-tabell fra teams.csv og masterlisten.
own_name_frames = []
for season in seasons:
    teams_path = DATA_DIR / season / "teams.csv"
    if teams_path.exists():
        names = pd.read_csv(teams_path, usecols=["id", "name"]).rename(
            columns={"id": "derived_team_id", "name": "derived_team_name"}
        )
        names["season_x"] = season
        own_name_frames.append(names)
master_path = DATA_DIR / "master_team_list.csv"
if master_path.exists():
    names = pd.read_csv(master_path)[["season", "team", "team_name"]].rename(
        columns={"season": "season_x", "team": "derived_team_id", "team_name": "derived_team_name"}
    )
    own_name_frames.append(names)
own_names = pd.concat(own_name_frames, ignore_index=True).drop_duplicates(
    ["season_x", "derived_team_id"]
)
all_seasons = all_seasons.merge(
    own_names, on=["season_x", "derived_team_id"], how="left", validate="many_to_one"
)
if "team" not in all_seasons.columns:
    all_seasons["team"] = all_seasons["derived_team_name"]
else:
    all_seasons["team"] = all_seasons["team"].fillna(all_seasons["derived_team_name"])
all_seasons = all_seasons.drop(columns=["derived_team_name"])
if "position" in all_seasons.columns:
    all_seasons["position"] = all_seasons.groupby("name")["position"].transform(
        lambda values: values.ffill().bfill()
    )

display(pd.DataFrame(load_report))
print(f"Totalt: {len(all_seasons):,} rader x {len(all_seasons.columns)} kolonner")

## 3. Legg til motstandernavn

Lag-ID-er er sesongspesifikke, så koblingen gjøres med både sesong og `opponent_team`.

In [ ]:
team_frames = []
for season in seasons:
    teams_path = DATA_DIR / season / "teams.csv"
    if not teams_path.exists():
        continue
    teams = pd.read_csv(teams_path, usecols=["id", "name"])
    teams = teams.rename(columns={"id": "opponent_team", "name": "opp_team_name"})
    teams["season_x"] = season
    team_frames.append(teams)

all_seasons = all_seasons.drop(columns=["opp_team_name"], errors="ignore")
if team_frames:
    opponent_lookup = pd.concat(team_frames, ignore_index=True).drop_duplicates(
        ["season_x", "opponent_team"]
    )
    all_seasons = all_seasons.merge(
        opponent_lookup, on=["season_x", "opponent_team"], how="left"
    )

# Reservekobling for eldre sesonger uten teams.csv.
master_path = DATA_DIR / "master_team_list.csv"
if master_path.exists():
    master = pd.read_csv(master_path)
    master_lookup = master[["season", "team", "team_name"]].rename(
        columns={"season": "season_x", "team": "opponent_team", "team_name": "fallback_name"}
    )
    master_lookup = master_lookup.drop_duplicates(["season_x", "opponent_team"])
    all_seasons = all_seasons.merge(
        master_lookup, on=["season_x", "opponent_team"], how="left"
    )
    all_seasons["opp_team_name"] = all_seasons["opp_team_name"].fillna(
        all_seasons["fallback_name"]
    )
    all_seasons = all_seasons.drop(columns="fallback_name")

print("Andel med motstandernavn:", all_seasons["opp_team_name"].notna().mean().round(3))

## 4. Beregn lagstatistikk

Vi lager først én rad per lag og kamp. Bruk av `fixture` gjør at double gameweeks behandles riktig.

In [ ]:
required = {"season_x", "team", "fixture", "GW", "kickoff_time", "was_home", "team_h_score", "team_a_score"}
missing = required.difference(all_seasons.columns)
if missing:
    raise ValueError(f"Mangler kolonner for lagaggregering: {sorted(missing)}")

team_matches = (
    all_seasons[list(required)]
    .drop_duplicates(["season_x", "team", "fixture"])
    .copy()
)

for column in ["team_h_score", "team_a_score"]:
    team_matches[column] = pd.to_numeric(team_matches[column], errors="coerce")

team_matches["team_goals_scored_match"] = team_matches["team_h_score"].where(
    team_matches["was_home"], team_matches["team_a_score"]
)
team_matches["team_goals_conceded_match"] = team_matches["team_a_score"].where(
    team_matches["was_home"], team_matches["team_h_score"]
)

played = team_matches["team_goals_scored_match"].notna() & team_matches["team_goals_conceded_match"].notna()
won = team_matches["team_goals_scored_match"] > team_matches["team_goals_conceded_match"]
drawn = team_matches["team_goals_scored_match"] == team_matches["team_goals_conceded_match"]
# Bruk float fra starten. pd.NA ville gitt object-dtype, som ikke støtter cumsum.
team_matches["points_match"] = float("nan")
team_matches.loc[played, "points_match"] = 0.0
team_matches.loc[played & drawn, "points_match"] = 1.0
team_matches.loc[played & won, "points_match"] = 3.0
team_matches["points_match"] = pd.to_numeric(
    team_matches["points_match"], errors="coerce"
)

team_matches = team_matches.sort_values(
    ["season_x", "team", "kickoff_time", "fixture"]
).reset_index(drop=True)

group_keys = [team_matches["season_x"], team_matches["team"]]
team_matches["points"] = team_matches["points_match"].fillna(0.0).groupby(group_keys).cumsum()
team_matches["team_goals_scored"] = pd.to_numeric(
    team_matches["team_goals_scored_match"], errors="coerce"
).fillna(0.0).groupby(group_keys).cumsum()
team_matches["team_goals_conceded"] = pd.to_numeric(
    team_matches["team_goals_conceded_match"], errors="coerce"
).fillna(0.0).groupby(group_keys).cumsum()
team_matches["team_goals_diff"] = (
    team_matches["team_goals_scored"] - team_matches["team_goals_conceded"]
)

display(team_matches.head())

## 5. Lag datalekkasjesikre før-kamp-variabler

Vanlige kumulative kolonner inkluderer den aktuelle kampen. `*_before_match` inneholder bare lagets resultater før kampen og er bedre egnet som modellvariabler.

In [ ]:
for source, target in {
    "points": "points_before_match",
    "team_goals_scored": "team_goals_scored_before_match",
    "team_goals_conceded": "team_goals_conceded_before_match",
    "team_goals_diff": "team_goals_diff_before_match",
}.items():
    team_matches[target] = (
        team_matches.groupby(["season_x", "team"], sort=False)[source]
        .shift(1)
        .fillna(0)
    )

display(
    team_matches[[
        "season_x", "team", "GW", "fixture", "points_match",
        "points", "points_before_match", "team_goals_diff_before_match"
    ]].head(20)
)

## 6. Koble lagdata tilbake til spillerradene

In [ ]:
aggregate_columns = [
    "points", "team_goals_scored", "team_goals_conceded", "team_goals_diff",
    "points_before_match", "team_goals_scored_before_match",
    "team_goals_conceded_before_match", "team_goals_diff_before_match",
]
all_seasons = all_seasons.drop(columns=aggregate_columns, errors="ignore")

team_lookup = team_matches[["season_x", "team", "fixture"] + aggregate_columns]
expanded = all_seasons.merge(
    team_lookup, on=["season_x", "team", "fixture"], how="left", validate="many_to_one"
)

if "value" in expanded.columns:
    expanded["price_m"] = pd.to_numeric(expanded["value"], errors="coerce") / 10

expanded = expanded.sort_values(
    ["season_x", "GW", "kickoff_time", "team", "name"]
).reset_index(drop=True)

print(f"Ferdig datasett: {len(expanded):,} rader x {len(expanded.columns)} kolonner")
display(expanded.head())

## 7. Valider og lagre

Filen lagres med et nytt navn. Originalen `cleaned_merged_seasons_team_aggregated.csv` blir ikke overskrevet.

In [ ]:
season_report = (
    expanded.groupby("season_x")
    .agg(rows=("name", "size"), players=("name", "nunique"), gameweeks=("GW", "nunique"))
    .reset_index()
)
display(season_report)

duplicate_keys = expanded.duplicated(
    ["season_x", "element", "fixture"], keep=False
).sum()
print("Duplikater på sesong + spiller + fixture:", duplicate_keys)
print("Andel med lagstatistikk:", expanded["points"].notna().mean().round(3))

OUTPUT_PATH = DATA_DIR / "cleaned_merged_seasons_team_aggregated_expanded.csv"
expanded.to_csv(OUTPUT_PATH, index=False)
print(f"Lagret {len(expanded):,} rader til:\n{OUTPUT_PATH}")